# Job Preprocessing Pipeline v3 — PhoBERT + Cosine Similarity

Mục tiêu:
- Chuẩn hoá dữ liệu job posting
- Trích xuất metadata + skills
- Tạo text tối ưu cho PhoBERT
- Sinh embedding cho matching / retrieval
- Dùng cosine similarity cho semantic search
- Xuất artifact cho matching, chatbot, section retrieval

In [ ]:
import re
import os
import json
import math
import html
import time
import unicodedata
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# Optional
try:
    from underthesea import word_tokenize
    HAS_UNDERTHESEA = True
except Exception:
    HAS_UNDERTHESEA = False

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

NOTEBOOK_VERSION = "preprocessing_v3_phobert"
RAW_INPUT_PATH = Path("/mnt/data/topcv_all_fields_merged_2026-03-16_20-57-23.xlsx")  # sửa lại nếu cần
ARTIFACT_DIR = Path("./artifacts_phobert_v3")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RUN_EMBEDDING = True
RUN_SECTION_EMBEDDING = True
SAVE_INTERMEDIATE = True

PHOBERT_MODEL_NAME = "vinai/phobert-base"
PHOBERT_MAX_LENGTH_MATCH = 256
PHOBERT_MAX_LENGTH_CHATBOT = 320
PHOBERT_MAX_LENGTH_CHUNK = 256
PHOBERT_BATCH_SIZE = 16
NORMALIZE_EMBEDDINGS = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("NOTEBOOK_VERSION:", NOTEBOOK_VERSION)
print("RAW_INPUT_PATH:", RAW_INPUT_PATH)
print("ARTIFACT_DIR:", ARTIFACT_DIR.resolve())
print("RUN_EMBEDDING:", RUN_EMBEDDING)
print("RUN_SECTION_EMBEDDING:", RUN_SECTION_EMBEDDING)
print("PHOBERT_MODEL_NAME:", PHOBERT_MODEL_NAME)
print("DEVICE:", DEVICE)
print("HAS_UNDERTHESEA:", HAS_UNDERTHESEA)

In [ ]:
def normalize_empty_value(val):
    if val is None:
        return None
    if isinstance(val, float) and pd.isna(val):
        return None

    s = str(val).strip()
    if not s:
        return None

    lowered = s.lower().strip()
    empty_tokens = {
        "nan", "none", "null", "n/a", "na", "-", "--", "---",
        "không", "không rõ", "chưa rõ", "chưa cập nhật", "đang cập nhật",
        "not specified", "unknown"
    }
    return None if lowered in empty_tokens else s


def safe_str(x):
    x = normalize_empty_value(x)
    return "" if x is None else str(x)


def get_series(df, col, default=None):
    if col in df.columns:
        return df[col]
    if default is None:
        default = [None] * len(df)
    return pd.Series(default, index=df.index)


def first_non_empty(*values):
    for v in values:
        v = normalize_empty_value(v)
        if v is not None:
            return v
    return None


def normalize_unicode(text):
    text = safe_str(text)
    return unicodedata.normalize("NFC", text)


def remove_html(text):
    text = safe_str(text)
    text = html.unescape(text)
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.I)
    text = re.sub(r"</p\s*>", "\n", text, flags=re.I)
    text = re.sub(r"<[^>]+>", " ", text)
    return text


def normalize_dash(text):
    text = safe_str(text)
    dash_map = {
        "–": "-",
        "—": "-",
        "−": "-",
        "•": "-",
        "●": "-",
        "▪": "-",
        "►": "-",
        "✅": "-",
        "✔": "-",
    }
    for k, v in dash_map.items():
        text = text.replace(k, v)
    return text


def deduplicate_list(values):
    out = []
    seen = set()
    for v in values:
        key = safe_str(v).strip()
        if not key:
            continue
        if key not in seen:
            seen.add(key)
            out.append(v)
    return out


def deduplicate_text_lines(text, min_key_len=12):
    text = safe_str(text)
    if not text:
        return ""
    out_lines = []
    seen = set()
    for line in text.splitlines():
        raw = line.strip()
        if not raw:
            continue
        key = re.sub(r"\s+", " ", raw.lower())
        if len(key) < min_key_len:
            out_lines.append(raw)
            continue
        if key not in seen:
            seen.add(key)
            out_lines.append(raw)
    return "\n".join(out_lines).strip()


def clean_text_light(text):
    text = normalize_empty_value(text)
    if text is None:
        return ""

    text = normalize_unicode(text)
    text = remove_html(text)
    text = normalize_dash(text)
    text = text.replace("\\n", "\n")
    text = re.sub(r"[\u200b-\u200f\uFEFF]", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r" ?\n ?", "\n", text)
    text = deduplicate_text_lines(text)
    return text.strip()


def clean_text_preserve_structure(text):
    text = clean_text_light(text)
    if not text:
        return ""
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n\s*-\s*", "\n- ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def clean_text_strict(text):
    text = clean_text_light(text)
    if not text:
        return ""
    text = text.lower()
    text = re.sub(r"[^\w\sàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ/+\-#\.]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def clean_text_for_phobert(text):
    text = normalize_empty_value(text)
    if text is None:
        return ""

    text = normalize_unicode(text)
    text = remove_html(text)
    text = normalize_dash(text)
    text = text.replace("\\n", "\n")

    text = re.sub(r"[\u200b-\u200f\uFEFF]", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    # giữ dấu câu đủ để preserve meaning cho PhoBERT
    text = re.sub(r"[^\w\sÀ-ỹ\.,;:/\-\+\#\(\)%\n]", " ", text)
    text = re.sub(r" +", " ", text)
    text = re.sub(r" ?\n ?", "\n", text)

    return text.strip()


def truncate_by_words(text, max_words=220):
    text = safe_str(text)
    if not text:
        return ""
    words = text.split()
    return " ".join(words[:max_words]).strip()


def save_table(df, base_path: Path):
    base_path = Path(base_path)
    try:
        out_path = str(base_path) + ".parquet"
        df.to_parquet(out_path, index=False)
        return out_path
    except Exception:
        out_path = str(base_path) + ".csv"
        df.to_csv(out_path, index=False, encoding="utf-8-sig")
        return out_path

In [ ]:
def load_raw_data(input_path: Path) -> pd.DataFrame:
    if not input_path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {input_path}")

    suffix = input_path.suffix.lower()
    if suffix in [".xlsx", ".xls"]:
        df = pd.read_excel(input_path)
    elif suffix == ".csv":
        df = pd.read_csv(input_path)
    else:
        raise ValueError(f"Unsupported file format: {suffix}")

    print(f"[INFO] Loaded raw data: {df.shape[0]} rows x {df.shape[1]} cols")
    return df


df_raw = load_raw_data(RAW_INPUT_PATH)
display(df_raw.head(3))
print(df_raw.columns.tolist())

In [ ]:
def merge_semantic_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    out["job_url"] = get_series(df, "job_url")
    out["job_id"] = get_series(df, "job_id")

    out["job_title_raw"] = [
        first_non_empty(a, b, c)
        for a, b, c in zip(
            get_series(df, "detail_title"),
            get_series(df, "title"),
            get_series(df, "job_title")
        )
    ]

    out["salary_raw"] = [
        first_non_empty(a, b, c)
        for a, b, c in zip(
            get_series(df, "detail_salary"),
            get_series(df, "salary_list"),
            get_series(df, "salary")
        )
    ]

    out["location_raw"] = [
        first_non_empty(a, b, c)
        for a, b, c in zip(
            get_series(df, "detail_location"),
            get_series(df, "location"),
            get_series(df, "city")
        )
    ]

    out["working_addresses_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "working_address"),
            get_series(df, "working_addresses")
        )
    ]

    out["working_times_raw"] = get_series(df, "working_times")

    out["experience_raw"] = [
        first_non_empty(a, b, c)
        for a, b, c in zip(
            get_series(df, "detail_experience"),
            get_series(df, "experience"),
            get_series(df, "years_of_experience")
        )
    ]

    out["education_level_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "education_level"),
            get_series(df, "degree")
        )
    ]

    out["employment_type_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "employment_type"),
            get_series(df, "job_type")
        )
    ]

    out["job_level_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "job_level"),
            get_series(df, "position_level")
        )
    ]

    out["description_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "description"),
            get_series(df, "detail_description")
        )
    ]

    out["requirements_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "requirements"),
            get_series(df, "detail_requirements")
        )
    ]

    out["benefits_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "benefits"),
            get_series(df, "detail_benefits")
        )
    ]

    out["tags_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "tags"),
            get_series(df, "tag_list")
        )
    ]

    out["deadline_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "deadline"),
            get_series(df, "expired_date")
        )
    ]

    out["company_name_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "company_name_full"),
            get_series(df, "company_name")
        )
    ]

    out["company_website_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "company_website"),
            get_series(df, "website")
        )
    ]

    out["company_field_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "company_field"),
            get_series(df, "industry")
        )
    ]

    out["company_description_raw"] = [
        first_non_empty(a, b)
        for a, b in zip(
            get_series(df, "company_description"),
            get_series(df, "company_overview")
        )
    ]

    return out


df = merge_semantic_columns(df_raw)
print("[INFO] After merging:", df.shape)
display(df.head(3))
df.info()

In [ ]:
def detect_language_type(text: str) -> str:
    text = safe_str(text)
    if not text:
        return "empty"

    has_vi = bool(re.search(r"[ăâêôơưđàáạảãèéẹẻẽìíịỉĩòóọỏõùúụủũỳýỵỷỹ]", text.lower()))
    has_en = bool(re.search(r"[a-z]", text.lower()))

    if has_vi and has_en:
        return "mixed"
    if has_vi:
        return "vi"
    if has_en:
        return "en"
    return "other"


audit_rows = []
combined_text = (
    get_series(df, "job_title_raw", "") .fillna("") + " " +
    get_series(df, "description_raw", "") .fillna("") + " " +
    get_series(df, "requirements_raw", "") .fillna("")
)

lang_type = combined_text.apply(detect_language_type)

audit_rows.append({
    "n_rows": len(df),
    "dup_by_url_ratio": df["job_url"].duplicated().mean() if "job_url" in df.columns else np.nan,
    "dup_soft_ratio": df["job_title_raw"].fillna("").duplicated().mean(),
    "has_minimum_content_ratio": (
        (df["job_title_raw"].fillna("").str.len() > 0) &
        (
            (df["description_raw"].fillna("").str.len() > 0) |
            (df["requirements_raw"].fillna("").str.len() > 0)
        )
    ).mean(),
    "mixed_ratio": (lang_type == "mixed").mean(),
    "en_ratio": (lang_type == "en").mean(),
    "vi_ratio": (lang_type == "vi").mean(),
})

audit_df = pd.DataFrame(audit_rows)
missing_df = pd.DataFrame({
    "column": df.columns,
    "missing_ratio": [df[c].isna().mean() for c in df.columns]
}).sort_values("missing_ratio", ascending=False)

display(audit_df)
display(missing_df)

In [ ]:
df_clean = df.copy()

for col in df_clean.columns:
    df_clean[col] = df_clean[col].apply(normalize_empty_value)

print("df_raw shape :", df.shape)
print("df_clean shape:", df_clean.shape)
display(df_clean.head(3))

In [ ]:
base_text_cols = [
    "job_title_raw",
    "salary_raw",
    "location_raw",
    "working_addresses_raw",
    "working_times_raw",
    "experience_raw",
    "education_level_raw",
    "employment_type_raw",
    "job_level_raw",
    "description_raw",
    "requirements_raw",
    "benefits_raw",
    "tags_raw",
    "deadline_raw",
    "company_name_raw",
    "company_website_raw",
    "company_field_raw",
    "company_description_raw",
]

for col in base_text_cols:
    if col in df_clean.columns:
        prefix = col.replace("_raw", "")
        df_clean[f"{prefix}_clean_light"] = df_clean[col].apply(clean_text_light)
        df_clean[f"{prefix}_clean_struct"] = df_clean[col].apply(clean_text_preserve_structure)
        df_clean[f"{prefix}_clean_strict"] = df_clean[col].apply(clean_text_strict)
        df_clean[f"{prefix}_clean_phobert"] = df_clean[col].apply(clean_text_for_phobert)

print("[INFO] Đã tạo xong các cột clean_*")
clean_cols = [c for c in df_clean.columns if "_clean_" in c]
print("Số cột clean:", len(clean_cols))
display(df_clean[[c for c in clean_cols[:12]]].head(2))

In [ ]:
preview_cols = [
    "job_title_raw", "job_title_clean_light", "job_title_clean_strict", "job_title_clean_phobert",
    "requirements_raw", "requirements_clean_struct", "requirements_clean_strict", "requirements_clean_phobert",
    "description_raw", "description_clean_light", "description_clean_phobert"
]
preview_cols = [c for c in preview_cols if c in df_clean.columns]

display(df_clean[preview_cols].head(3))

empty_ratio_df = pd.DataFrame({
    "column": preview_cols,
    "empty_ratio": [(df_clean[c].fillna("").str.strip() == "").mean() for c in preview_cols]
})
display(empty_ratio_df)

In [ ]:
TITLE_TRAILING_NOISE_PATTERNS = [
    r"\|\s*.*$",
    r"\-\s*lương.*$",
    r"\-\s*thu nhập.*$",
    r"\-\s*hà nội.*$",
    r"\-\s*hồ chí minh.*$",
    r"\-\s*đà nẵng.*$",
    r"\(\s*urgent\s*\)",
    r"\(\s*hot\s*\)",
    r"\(\s*remote\s*\)",
    r"\(\s*hybrid\s*\)",
    r"\(\s*onsite\s*\)",
]

BRACKET_NOISE_KEYWORDS = {
    "hà nội", "ha noi", "hcm", "tp hcm", "hồ chí minh", "da nang",
    "remote", "hybrid", "onsite", "urgent", "hot"
}

TITLE_SYNONYM_MAP = {
    "chuyên viên phân tích dữ liệu": "data analyst",
    "nhân viên phân tích dữ liệu": "data analyst",
    "data analytics engineer": "analytics engineer",
    "bi analyst": "business intelligence analyst",
    "business analyst": "business analyst",
    "fp&a analyst": "fp&a analyst",
    "ml engineer": "machine learning engineer",
    "ai engineer": "ai engineer",
    "data engineer": "data engineer",
    "data scientist": "data scientist",
    "data governance specialist": "data governance specialist",
    "data quality analyst": "data quality analyst",
    "product analyst": "product analyst",
}

JOB_FAMILY_RULES = {
    "data_analytics": [
        "data analyst", "business intelligence analyst", "analytics engineer",
        "product analyst", "fp&a analyst", "bi analyst"
    ],
    "data_engineering": [
        "data engineer", "etl developer", "big data engineer"
    ],
    "data_science_ml": [
        "data scientist", "machine learning engineer", "ai engineer", "ml engineer"
    ],
    "data_governance_quality": [
        "data governance specialist", "data quality analyst", "data steward"
    ],
    "product_project_ba": [
        "business analyst", "project manager", "product manager"
    ],
    "software_engineering": [
        "backend developer", "frontend developer", "fullstack developer", "software engineer"
    ],
}

SENIORITY_RULES = {
    "director": [r"\bdirector\b", r"\bhead\b"],
    "manager": [r"\bmanager\b", r"trưởng"],
    "lead": [r"\blead\b", r"team lead"],
    "senior": [r"\bsenior\b", r"\bsr\b"],
    "middle": [r"\bmid\b", r"\bmiddle\b"],
    "junior": [r"\bjunior\b", r"\bjr\b"],
    "intern": [r"\bintern\b", r"thực tập"],
    "fresher": [r"\bfresher\b"],
}


def strip_bracket_noise(text):
    text = safe_str(text)
    if not text:
        return ""
    matches = re.findall(r"\((.*?)\)", text)
    for m in matches:
        normalized = clean_text_strict(m)
        if normalized in BRACKET_NOISE_KEYWORDS:
            text = text.replace(f"({m})", " ")
    return re.sub(r"\s+", " ", text).strip()


def normalize_job_title(text):
    text = clean_text_light(text)
    if not text:
        return ""

    text = strip_bracket_noise(text)
    for pat in TITLE_TRAILING_NOISE_PATTERNS:
        text = re.sub(pat, " ", text, flags=re.I)

    text = re.sub(r"\s+", " ", text).strip(" -|")
    lowered = text.lower().strip()

    if lowered in TITLE_SYNONYM_MAP:
        return TITLE_SYNONYM_MAP[lowered]

    return lowered


def infer_job_family(job_title_canonical):
    t = safe_str(job_title_canonical).lower()
    for family, keywords in JOB_FAMILY_RULES.items():
        if any(k in t for k in keywords):
            return family
    return "unknown"


def infer_seniority_from_text(text):
    t = clean_text_strict(text)
    if not t:
        return "unknown"
    for level, patterns in SENIORITY_RULES.items():
        for p in patterns:
            if re.search(p, t, flags=re.I):
                return level
    return "unknown"

In [ ]:
df_clean["job_title_display"] = df_clean["job_title_clean_light"].fillna("")
df_clean["job_title_canonical"] = df_clean["job_title_raw"].apply(normalize_job_title)
df_clean["job_family"] = df_clean["job_title_canonical"].apply(infer_job_family)
df_clean["seniority_from_title"] = df_clean["job_title_raw"].apply(infer_seniority_from_text)

df_clean["job_title_for_phobert"] = df_clean["job_title_display"].where(
    df_clean["job_title_display"].fillna("").str.strip() != "",
    df_clean["job_title_canonical"]
).apply(clean_text_for_phobert)

display(
    df_clean[
        ["job_title_raw", "job_title_display", "job_title_canonical", "job_family", "seniority_from_title", "job_title_for_phobert"]
    ].head(10)
)

In [ ]:
VIETNAM_CITY_ALIASES = {
    "hà nội": "Hà Nội",
    "ha noi": "Hà Nội",
    "hn": "Hà Nội",
    "hồ chí minh": "Hồ Chí Minh",
    "ho chi minh": "Hồ Chí Minh",
    "tp hcm": "Hồ Chí Minh",
    "hcm": "Hồ Chí Minh",
    "sài gòn": "Hồ Chí Minh",
    "sai gon": "Hồ Chí Minh",
    "đà nẵng": "Đà Nẵng",
    "da nang": "Đà Nẵng",
    "hải phòng": "Hải Phòng",
    "hai phong": "Hải Phòng",
    "cần thơ": "Cần Thơ",
    "can tho": "Cần Thơ",
}

WORK_MODE_RULES = {
    "remote": [r"\bremote\b", r"làm việc từ xa", r"work from home", r"\bwfh\b"],
    "hybrid": [r"\bhybrid\b", r"linh hoạt", r"kết hợp onsite và remote"],
    "onsite": [r"\bonsite\b", r"tại văn phòng", r"làm việc tại công ty"],
}


def detect_city_from_text(text):
    t = clean_text_strict(text)
    for alias, canonical in VIETNAM_CITY_ALIASES.items():
        if alias in t:
            return canonical
    return None


def infer_work_mode(*texts):
    merged = " ".join([clean_text_strict(t) for t in texts if safe_str(t)])
    if not merged:
        return "unknown"
    for mode, patterns in WORK_MODE_RULES.items():
        for p in patterns:
            if re.search(p, merged, flags=re.I):
                return mode
    return "unknown"


def has_multi_location(text):
    t = clean_text_strict(text)
    hits = set()
    for alias, canonical in VIETNAM_CITY_ALIASES.items():
        if alias in t:
            hits.add(canonical)
    return len(hits) >= 2


def parse_working_address(raw_text):
    text = clean_text_light(raw_text)
    city = detect_city_from_text(text)
    district = None
    m = re.search(r"quận\s+(\d+|[a-zà-ỹ]+)", text, flags=re.I)
    if m:
        district = m.group(0).strip()

    return {
        "working_address_clean": text,
        "location_city": city,
        "location_district": district,
        "is_multi_location": has_multi_location(text)
    }


def normalize_location(location_raw, working_addresses_raw):
    city_1 = detect_city_from_text(location_raw)
    city_2 = detect_city_from_text(working_addresses_raw)
    return first_non_empty(city_1, city_2, clean_text_light(location_raw), clean_text_light(working_addresses_raw))


def parse_deadline(raw, reference_date=None):
    reference_date = reference_date or datetime.today().date()
    text = clean_text_light(raw)

    if not text:
        return {
            "deadline_clean": "",
            "deadline_date": None,
            "days_to_deadline": None,
            "deadline_type": "unknown",
            "is_expired": None,
        }

    # dd/mm/yyyy or dd-mm-yyyy
    m = re.search(r"(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})", text)
    if m:
        day, month, year = map(int, m.groups())
        if year < 100:
            year += 2000
        try:
            dt = datetime(year, month, day).date()
            return {
                "deadline_clean": text,
                "deadline_date": str(dt),
                "days_to_deadline": (dt - reference_date).days,
                "deadline_type": "absolute_date",
                "is_expired": dt < reference_date,
            }
        except Exception:
            pass

    m = re.search(r"(\d+)\s*ngày", clean_text_strict(text))
    if m:
        days = int(m.group(1))
        dt = reference_date + timedelta(days=days)
        return {
            "deadline_clean": text,
            "deadline_date": str(dt),
            "days_to_deadline": days,
            "deadline_type": "relative_days",
            "is_expired": False,
        }

    return {
        "deadline_clean": text,
        "deadline_date": None,
        "days_to_deadline": None,
        "deadline_type": "unknown",
        "is_expired": None,
    }

In [ ]:
address_parsed = df_clean["working_addresses_raw"].apply(parse_working_address)
address_df = pd.DataFrame(address_parsed.tolist(), index=df_clean.index)

for c in address_df.columns:
    df_clean[c] = address_df[c]

df_clean["location_norm"] = [
    normalize_location(a, b)
    for a, b in zip(df_clean["location_raw"], df_clean["working_addresses_raw"])
]

df_clean["work_mode"] = [
    infer_work_mode(a, b, c, d)
    for a, b, c, d in zip(
        df_clean["job_title_raw"],
        df_clean["location_raw"],
        df_clean["working_addresses_raw"],
        df_clean["description_raw"]
    )
]

deadline_parsed = df_clean["deadline_raw"].apply(parse_deadline)
deadline_df = pd.DataFrame(deadline_parsed.tolist(), index=df_clean.index)
for c in deadline_df.columns:
    df_clean[c] = deadline_df[c]

display(df_clean[
    [
        "location_raw", "working_addresses_raw", "location_city",
        "location_district", "location_norm", "work_mode",
        "deadline_raw", "deadline_date", "days_to_deadline", "deadline_type", "is_expired"
    ]
].head(10))

In [ ]:
def clean_salary(raw):
    text = clean_text_light(raw).lower()
    text = text.replace("thoả", "thỏa")
    return text


def parse_numeric_token(num_text):
    s = safe_str(num_text).strip().replace(" ", "")
    if not s:
        return None

    if "," in s and "." in s:
        # chọn heuristic đơn giản
        s = s.replace(".", "").replace(",", ".")
    elif s.count(".") > 1:
        s = s.replace(".", "")
    elif s.count(",") > 1:
        s = s.replace(",", "")
    else:
        s = s.replace(",", ".")

    try:
        return float(s)
    except Exception:
        return None


def detect_salary_multiplier(text, currency):
    t = text.lower()
    if currency == "usd":
        return 1.0
    if "triệu" in t or "trieu" in t:
        return 1_000_000
    if "nghìn" in t or "ngan" in t or "k" in t:
        return 1_000
    return 1.0


def parse_salary_range(raw):
    text = clean_salary(raw)
    if not text:
        return {
            "salary_clean": "",
            "salary_min_value": None,
            "salary_max_value": None,
            "salary_currency": "unknown",
            "salary_period": "month",
            "salary_type": "unknown",
            "salary_is_negotiable": None,
            "salary_min_vnd_month": None,
            "salary_max_vnd_month": None,
        }

    currency = "usd" if "usd" in text or "$" in text else "vnd"
    negotiable = ("thỏa thuận" in text) or ("negotiable" in text)

    nums = re.findall(r"\d+(?:[.,]\d+)?", text)
    nums = [parse_numeric_token(x) for x in nums]
    nums = [x for x in nums if x is not None]

    multiplier = detect_salary_multiplier(text, currency)

    salary_min = None
    salary_max = None
    salary_type = "unknown"

    if negotiable and not nums:
        salary_type = "negotiable"
    elif "up to" in text or "tối đa" in text:
        if nums:
            salary_max = nums[0] * multiplier
            salary_type = "upper_bound"
    elif len(nums) >= 2:
        salary_min = nums[0] * multiplier
        salary_max = nums[1] * multiplier
        salary_type = "range"
    elif len(nums) == 1:
        salary_min = nums[0] * multiplier
        salary_max = nums[0] * multiplier
        salary_type = "fixed"

    if currency == "usd":
        # quy đổi tạm
        fx = 25000
        if salary_min is not None:
            salary_min = salary_min * fx
        if salary_max is not None:
            salary_max = salary_max * fx

    return {
        "salary_clean": text,
        "salary_min_value": salary_min,
        "salary_max_value": salary_max,
        "salary_currency": currency,
        "salary_period": "month",
        "salary_type": salary_type,
        "salary_is_negotiable": negotiable,
        "salary_min_vnd_month": salary_min,
        "salary_max_vnd_month": salary_max,
    }


salary_parsed = df_clean["salary_raw"].apply(parse_salary_range)
salary_df = pd.DataFrame(salary_parsed.tolist(), index=df_clean.index)
for c in salary_df.columns:
    df_clean[c] = salary_df[c]

display(df_clean[
    [
        "salary_raw", "salary_clean", "salary_min_value", "salary_max_value",
        "salary_currency", "salary_type", "salary_is_negotiable",
        "salary_min_vnd_month", "salary_max_vnd_month"
    ]
].head(10))

In [ ]:
def clean_experience(text):
    return clean_text_strict(text)


def parse_experience_range(raw):
    text = clean_experience(raw)
    if not text:
        return {
            "experience_clean": "",
            "experience_min_years": None,
            "experience_max_years": None,
            "experience_type": "unknown",
        }

    if "không yêu cầu" in text or "no experience" in text:
        return {
            "experience_clean": text,
            "experience_min_years": 0.0,
            "experience_max_years": 0.0,
            "experience_type": "no_experience",
        }

    m_month = re.search(r"(\d+(?:\.\d+)?)\s*tháng", text)
    if m_month:
        months = float(m_month.group(1))
        years = months / 12.0
        return {
            "experience_clean": text,
            "experience_min_years": years,
            "experience_max_years": years,
            "experience_type": "fixed",
        }

    nums = re.findall(r"\d+(?:\.\d+)?", text)
    nums = [float(x) for x in nums] if nums else []

    if len(nums) >= 2:
        return {
            "experience_clean": text,
            "experience_min_years": nums[0],
            "experience_max_years": nums[1],
            "experience_type": "range",
        }

    if len(nums) == 1:
        val = nums[0]
        if "từ" in text or "+" in text or "trên" in text or "ít nhất" in text:
            return {
                "experience_clean": text,
                "experience_min_years": val,
                "experience_max_years": None,
                "experience_type": "minimum",
            }
        if "dưới" in text:
            return {
                "experience_clean": text,
                "experience_min_years": 0.0,
                "experience_max_years": val,
                "experience_type": "maximum",
            }
        return {
            "experience_clean": text,
            "experience_min_years": val,
            "experience_max_years": val,
            "experience_type": "fixed",
        }

    return {
        "experience_clean": text,
        "experience_min_years": None,
        "experience_max_years": None,
        "experience_type": "unknown",
    }


EDUCATION_MAP = {
    "phd": ["tiến sĩ", "phd", "doctor"],
    "master": ["thạc sĩ", "master"],
    "bachelor": ["đại học", "cử nhân", "bachelor"],
    "college": ["cao đẳng", "college"],
    "high_school": ["trung học", "high school"],
}

EMPLOYMENT_TYPE_MAP = {
    "full_time": ["toàn thời gian", "full time", "full-time"],
    "part_time": ["bán thời gian", "part time", "part-time"],
    "internship": ["thực tập", "internship", "intern"],
    "contract": ["hợp đồng", "contract"],
    "freelance": ["freelance", "cộng tác viên"],
    "temporary": ["temporary", "thời vụ"],
}


def normalize_education_level(text):
    t = clean_text_strict(text)
    for level, kws in EDUCATION_MAP.items():
        if any(k in t for k in kws):
            return level
    return "unknown"


def normalize_employment_type(text):
    t = clean_text_strict(text)
    for level, kws in EMPLOYMENT_TYPE_MAP.items():
        if any(k in t for k in kws):
            return level
    return "unknown"


JOB_LEVEL_RULES = {
    "director": [r"\bdirector\b", r"\bhead\b"],
    "manager": [r"\bmanager\b", r"trưởng"],
    "lead": [r"\blead\b"],
    "senior": [r"\bsenior\b", r"\bsr\b"],
    "middle": [r"\bmiddle\b", r"\bmid\b"],
    "junior": [r"\bjunior\b", r"\bjr\b"],
    "fresher": [r"\bfresher\b"],
    "intern": [r"\bintern\b", r"thực tập"],
}


def normalize_job_level(text, title_text=None):
    merged = clean_text_strict(safe_str(text) + " " + safe_str(title_text))
    for lvl, patterns in JOB_LEVEL_RULES.items():
        for p in patterns:
            if re.search(p, merged, flags=re.I):
                return lvl
    return "unknown"


exp_parsed = df_clean["experience_raw"].apply(parse_experience_range)
exp_df = pd.DataFrame(exp_parsed.tolist(), index=df_clean.index)
for c in exp_df.columns:
    df_clean[c] = exp_df[c]

df_clean["education_level_norm"] = df_clean["education_level_raw"].apply(normalize_education_level)
df_clean["employment_type_norm"] = df_clean["employment_type_raw"].apply(normalize_employment_type)
df_clean["job_level_norm"] = [
    normalize_job_level(a, b)
    for a, b in zip(df_clean["job_level_raw"], df_clean["job_title_raw"])
]

display(df_clean[
    [
        "experience_raw", "experience_min_years", "experience_max_years", "experience_type",
        "education_level_raw", "education_level_norm",
        "employment_type_raw", "employment_type_norm",
        "job_level_raw", "job_level_norm"
    ]
].head(10))

In [ ]:
def normalize_tags(text):
    text = clean_text_light(text)
    if not text:
        return []
    parts = re.split(r"[,\|;/\n]+", text)
    parts = [clean_text_strict(p) for p in parts]
    parts = [p for p in parts if p]
    return deduplicate_list(parts)


df_clean["tags_list"] = df_clean["tags_raw"].apply(normalize_tags)
df_clean["tags_text_phobert"] = df_clean["tags_list"].apply(
    lambda xs: ", ".join(xs) if isinstance(xs, list) else ""
).apply(clean_text_for_phobert)

display(df_clean[["tags_raw", "tags_list", "tags_text_phobert"]].head(10))

In [ ]:
SKILL_TAXONOMY = {
    "python": {"aliases": ["python"], "group": "programming"},
    "sql": {"aliases": ["sql", "postgresql", "mysql", "sql server"], "group": "database"},
    "excel": {"aliases": ["excel", "microsoft excel"], "group": "analytics_tools"},
    "power bi": {"aliases": ["power bi", "powerbi"], "group": "bi_tools"},
    "tableau": {"aliases": ["tableau"], "group": "bi_tools"},
    "pandas": {"aliases": ["pandas"], "group": "python_libs"},
    "numpy": {"aliases": ["numpy"], "group": "python_libs"},
    "scikit-learn": {"aliases": ["scikit-learn", "sklearn"], "group": "ml_libs"},
    "tensorflow": {"aliases": ["tensorflow"], "group": "ml_libs"},
    "pytorch": {"aliases": ["pytorch", "torch"], "group": "ml_libs"},
    "machine learning": {"aliases": ["machine learning", "ml"], "group": "ml_concepts"},
    "deep learning": {"aliases": ["deep learning", "dl"], "group": "ml_concepts"},
    "statistics": {"aliases": ["statistics", "thống kê"], "group": "analytics_concepts"},
    "etl": {"aliases": ["etl"], "group": "data_engineering"},
    "airflow": {"aliases": ["airflow", "apache airflow"], "group": "data_engineering"},
    "spark": {"aliases": ["spark", "apache spark", "pyspark"], "group": "data_engineering"},
    "hadoop": {"aliases": ["hadoop"], "group": "data_engineering"},
    "kafka": {"aliases": ["kafka", "apache kafka"], "group": "data_engineering"},
    "aws": {"aliases": ["aws", "amazon web services"], "group": "cloud"},
    "gcp": {"aliases": ["gcp", "google cloud platform"], "group": "cloud"},
    "azure": {"aliases": ["azure"], "group": "cloud"},
    "docker": {"aliases": ["docker"], "group": "devops"},
    "kubernetes": {"aliases": ["kubernetes", "k8s"], "group": "devops"},
    "git": {"aliases": ["git", "github", "gitlab"], "group": "dev_tools"},
    "api": {"aliases": ["api", "rest api", "restful api"], "group": "backend"},
    "fastapi": {"aliases": ["fastapi"], "group": "backend"},
    "flask": {"aliases": ["flask"], "group": "backend"},
    "llm": {"aliases": ["llm", "large language model", "large language models"], "group": "ai"},
    "nlp": {"aliases": ["nlp", "natural language processing"], "group": "ai"},
    "computer vision": {"aliases": ["computer vision", "cv"], "group": "ai"},
    "data modeling": {"aliases": ["data modeling", "data model"], "group": "data_modeling"},
    "data warehouse": {"aliases": ["data warehouse", "dwh"], "group": "data_modeling"},
    "bigquery": {"aliases": ["bigquery"], "group": "database"},
    "snowflake": {"aliases": ["snowflake"], "group": "database"},
    "oracle": {"aliases": ["oracle"], "group": "database"},
    "sas": {"aliases": ["sas"], "group": "analytics_tools"},
    "r": {"aliases": [" r ", "(r)", "ngôn ngữ r"], "group": "programming"},
    "communication": {"aliases": ["communication", "giao tiếp"], "group": "soft_skills"},
    "problem solving": {"aliases": ["problem solving", "giải quyết vấn đề"], "group": "soft_skills"},
}

In [ ]:
REQUIRED_HINTS = [
    "bắt buộc", "required", "must have", "cần có", "thành thạo", "proficient", "kinh nghiệm với"
]
PREFERRED_HINTS = [
    "ưu tiên", "preferred", "nice to have", "là lợi thế", "plus point"
]


def alias_to_regex(alias):
    alias = safe_str(alias)
    if not alias:
        return None

    alias_strict = alias.strip()
    if alias_strict.lower() == "r":
        return r"(?<!\w)r(?!\w)"
    if alias_strict.lower() == "ml":
        return r"(?<!\w)ml(?!\w)"
    if alias_strict.lower() == "cv":
        return r"(?<!\w)cv(?!\w)"
    return r"(?<!\w)" + re.escape(alias_strict) + r"(?!\w)"


SKILL_PATTERNS = {}
for skill, meta in SKILL_TAXONOMY.items():
    pats = []
    for alias in meta["aliases"]:
        pat = alias_to_regex(alias)
        if pat:
            pats.append(re.compile(pat, flags=re.I))
    SKILL_PATTERNS[skill] = pats


def infer_skill_importance(segment, source_field):
    s = clean_text_strict(segment)
    if any(h in s for h in PREFERRED_HINTS):
        return "preferred"
    if source_field == "requirements":
        return "required"
    if any(h in s for h in REQUIRED_HINTS):
        return "required"
    return "mentioned"


def extract_skill_records_from_text(text, source_field="unknown"):
    text = clean_text_preserve_structure(text)
    if not text:
        return []

    segments = [seg.strip() for seg in re.split(r"[\n•\-;]+", text) if seg.strip()]
    records = []

    for seg in segments:
        for skill, patterns in SKILL_PATTERNS.items():
            matched = False
            for p in patterns:
                if p.search(seg):
                    matched = True
                    break
            if matched:
                records.append({
                    "skill": skill,
                    "skill_group": SKILL_TAXONOMY[skill]["group"],
                    "source_field": source_field,
                    "importance": infer_skill_importance(seg, source_field),
                    "excerpt": seg[:300]
                })

    # dedup
    seen = set()
    out = []
    for r in records:
        key = (r["skill"], r["source_field"], r["importance"])
        if key not in seen:
            seen.add(key)
            out.append(r)
    return out


def merge_skill_records(*record_lists):
    merged = []
    seen = set()
    for records in record_lists:
        for r in records:
            key = (r["skill"], r["source_field"], r["importance"])
            if key not in seen:
                seen.add(key)
                merged.append(r)
    return merged


def list_from_records(records, key, importance_filter=None):
    vals = []
    for r in records:
        if importance_filter and r.get("importance") != importance_filter:
            continue
        if r.get(key):
            vals.append(r[key])
    return deduplicate_list(vals)

In [ ]:
title_skill_records = df_clean["job_title_clean_phobert"].apply(lambda x: extract_skill_records_from_text(x, "title"))
tag_skill_records = df_clean["tags_text_phobert"].apply(lambda x: extract_skill_records_from_text(x, "tags"))
req_skill_records = df_clean["requirements_clean_phobert"].apply(lambda x: extract_skill_records_from_text(x, "requirements"))
desc_skill_records = df_clean["description_clean_phobert"].apply(lambda x: extract_skill_records_from_text(x, "description"))

df_clean["skill_records"] = [
    merge_skill_records(a, b, c, d)
    for a, b, c, d in zip(title_skill_records, tag_skill_records, req_skill_records, desc_skill_records)
]

df_clean["skills_extracted"] = df_clean["skill_records"].apply(lambda rs: list_from_records(rs, "skill"))
df_clean["skills_required"] = df_clean["skill_records"].apply(lambda rs: list_from_records(rs, "skill", "required"))
df_clean["skills_preferred"] = df_clean["skill_records"].apply(lambda rs: list_from_records(rs, "skill", "preferred"))
df_clean["skill_groups"] = df_clean["skill_records"].apply(lambda rs: list_from_records(rs, "skill_group"))

df_clean["skills_text_phobert"] = df_clean["skills_extracted"].apply(
    lambda xs: ", ".join(xs) if isinstance(xs, list) else ""
)
df_clean["skills_required_text_phobert"] = df_clean["skills_required"].apply(
    lambda xs: ", ".join(xs) if isinstance(xs, list) else ""
)
df_clean["skills_preferred_text_phobert"] = df_clean["skills_preferred"].apply(
    lambda xs: ", ".join(xs) if isinstance(xs, list) else ""
)

display(df_clean[
    [
        "job_title_raw", "skills_extracted", "skills_required",
        "skills_preferred", "skill_groups",
        "skills_text_phobert", "skills_required_text_phobert"
    ]
].head(10))

In [ ]:
job_skill_rows = []
for _, row in df_clean.iterrows():
    for r in row["skill_records"]:
        job_skill_rows.append({
            "job_url": row.get("job_url"),
            "job_title_display": row.get("job_title_display"),
            "job_title_canonical": row.get("job_title_canonical"),
            "job_family": row.get("job_family"),
            "location_norm": row.get("location_norm"),
            "skill": r["skill"],
            "skill_group": r["skill_group"],
            "source_field": r["source_field"],
            "importance": r["importance"],
            "excerpt": r["excerpt"],
        })

job_skill_map_df = pd.DataFrame(job_skill_rows)
display(job_skill_map_df.head(10))

if len(job_skill_map_df) > 0:
    role_skill_stats_df = (
        job_skill_map_df.groupby(["job_family", "skill", "importance"])
        .size()
        .reset_index(name="job_count")
        .sort_values(["job_family", "job_count"], ascending=[True, False])
    )
else:
    role_skill_stats_df = pd.DataFrame(columns=["job_family", "skill", "importance", "job_count"])

display(role_skill_stats_df.head(20))
print("Tỷ lệ job không extract được skill:", (df_clean["skills_extracted"].apply(len) == 0).mean())

In [ ]:
def format_salary_brief(row):
    mn = row.get("salary_min_vnd_month")
    mx = row.get("salary_max_vnd_month")
    if pd.notna(mn) and pd.notna(mx):
        return f"{int(mn):,}-{int(mx):,} VND/tháng"
    if pd.notna(mn):
        return f"từ {int(mn):,} VND/tháng"
    if row.get("salary_is_negotiable") is True:
        return "thỏa thuận"
    return ""


def build_job_text_sparse(row):
    parts = []

    for field in [
        row.get("job_title_canonical"),
        row.get("job_family"),
        row.get("tags_text_phobert"),
        row.get("skills_text_phobert"),
        row.get("skills_required_text_phobert"),
        row.get("requirements_clean_strict"),
        row.get("description_clean_strict"),
    ]:
        s = safe_str(field)
        if s:
            parts.append(s)

    return "\n".join(parts).strip()


def build_job_text_phobert_match(row):
    parts = []

    title = row.get("job_title_for_phobert")
    family = row.get("job_family")
    required_skills = row.get("skills_required_text_phobert")
    preferred_skills = row.get("skills_preferred_text_phobert")
    exp = row.get("experience_min_years")
    location = row.get("location_norm")
    work_mode = row.get("work_mode")
    req = truncate_by_words(row.get("requirements_clean_phobert"), 160)

    if title:
        parts.append(f"Vị trí: {title}")
    if family and family != "unknown":
        parts.append(f"Nhóm nghề: {family}")
    if required_skills:
        parts.append(f"Kỹ năng bắt buộc: {required_skills}")
    if preferred_skills:
        parts.append(f"Kỹ năng ưu tiên: {preferred_skills}")
    if exp is not None and not pd.isna(exp):
        parts.append(f"Kinh nghiệm tối thiểu: {exp} năm")
    if location:
        parts.append(f"Địa điểm: {location}")
    if work_mode and work_mode != "unknown":
        parts.append(f"Hình thức làm việc: {work_mode}")
    if req:
        parts.append(f"Yêu cầu chính: {req}")

    return "\n".join(parts).strip()


def build_job_text_phobert_chatbot(row):
    parts = []

    if row.get("job_title_for_phobert"):
        parts.append(f"Vị trí tuyển dụng: {row['job_title_for_phobert']}")
    if row.get("job_family") and row["job_family"] != "unknown":
        parts.append(f"Nhóm công việc: {row['job_family']}")
    if row.get("location_norm"):
        parts.append(f"Địa điểm: {row['location_norm']}")
    if row.get("work_mode") and row["work_mode"] != "unknown":
        parts.append(f"Hình thức làm việc: {row['work_mode']}")
    salary_brief = format_salary_brief(row)
    if salary_brief:
        parts.append(f"Mức lương: {salary_brief}")
    if row.get("skills_required_text_phobert"):
        parts.append(f"Kỹ năng bắt buộc: {row['skills_required_text_phobert']}")
    if row.get("skills_preferred_text_phobert"):
        parts.append(f"Kỹ năng ưu tiên: {row['skills_preferred_text_phobert']}")
    if row.get("requirements_clean_phobert"):
        parts.append(f"Yêu cầu:\n{truncate_by_words(row['requirements_clean_phobert'], 220)}")
    if row.get("description_clean_phobert"):
        parts.append(f"Mô tả công việc:\n{truncate_by_words(row['description_clean_phobert'], 220)}")
    if row.get("benefits_clean_phobert"):
        parts.append(f"Quyền lợi:\n{truncate_by_words(row['benefits_clean_phobert'], 120)}")

    return "\n\n".join(parts).strip()

In [ ]:
df_clean["job_text_sparse"] = df_clean.apply(build_job_text_sparse, axis=1)
df_clean["job_text_phobert_match"] = df_clean.apply(build_job_text_phobert_match, axis=1)
df_clean["job_text_phobert_chatbot"] = df_clean.apply(build_job_text_phobert_chatbot, axis=1)

df_clean["dense_encoder_route"] = "phobert"
df_clean["dense_similarity_metric"] = "cosine"

display(df_clean[
    [
        "job_title_raw", "job_text_sparse",
        "job_text_phobert_match", "job_text_phobert_chatbot"
    ]
].head(3))

In [ ]:
def split_long_text(text, max_chars=700, overlap=80):
    text = clean_text_preserve_structure(text)
    if not text:
        return []

    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks = []
    current = ""

    for para in paras:
        if len(current) + len(para) + 2 <= max_chars:
            current = f"{current}\n\n{para}".strip()
        else:
            if current:
                chunks.append(current)
            if len(para) <= max_chars:
                current = para
            else:
                # hard split
                start = 0
                while start < len(para):
                    end = min(start + max_chars, len(para))
                    chunk = para[start:end]
                    chunks.append(chunk.strip())
                    start = max(end - overlap, end)
                current = ""

    if current:
        chunks.append(current)

    return [c.strip() for c in chunks if c.strip()]


SECTION_PRIORITY = {
    "title": 5.0,
    "requirements": 4.5,
    "description": 4.0,
    "benefits": 3.0,
    "company": 2.5,
}


def build_chunk_text_phobert(row, section_type: str, chunk_text: str):
    title = safe_str(row.get("job_title_for_phobert"))
    family = safe_str(row.get("job_family"))
    section_map = {
        "title": "Tiêu đề",
        "requirements": "Yêu cầu",
        "description": "Mô tả công việc",
        "benefits": "Quyền lợi",
        "company": "Giới thiệu công ty",
    }
    section_label = section_map.get(section_type, section_type)

    parts = []
    if title:
        parts.append(f"Vị trí: {title}")
    if family and family != "unknown":
        parts.append(f"Nhóm nghề: {family}")
    parts.append(f"Phần: {section_label}")
    parts.append(truncate_by_words(chunk_text, 180))
    return "\n".join([p for p in parts if p]).strip()


def build_job_section_records(row):
    rows = []

    section_map = {
        "title": safe_str(row.get("job_title_for_phobert")),
        "requirements": safe_str(row.get("requirements_clean_phobert")),
        "description": safe_str(row.get("description_clean_phobert")),
        "benefits": safe_str(row.get("benefits_clean_phobert")),
        "company": safe_str(row.get("company_description_clean_phobert")),
    }

    for section_type, text in section_map.items():
        if not text:
            continue

        chunks = [text] if section_type == "title" else split_long_text(text, max_chars=700, overlap=80)
        for chunk_order, chunk_text in enumerate(chunks):
            rows.append({
                "job_url": row.get("job_url"),
                "job_title_display": row.get("job_title_display"),
                "job_title_canonical": row.get("job_title_canonical"),
                "job_family": row.get("job_family"),
                "location_norm": row.get("location_norm"),
                "work_mode": row.get("work_mode"),
                "section_type": section_type,
                "chunk_order": chunk_order,
                "section_priority": SECTION_PRIORITY.get(section_type, 1.0),
                "chunk_text_raw": chunk_text,
                "chunk_text_phobert": build_chunk_text_phobert(row, section_type, chunk_text),
            })

    return rows


section_rows = []
for _, row in df_clean.iterrows():
    section_rows.extend(build_job_section_records(row))

job_sections_df = pd.DataFrame(section_rows)
display(job_sections_df.head(10))
print("job_sections_df shape:", job_sections_df.shape)

In [ ]:
def maybe_segment_vi_text(text):
    text = safe_str(text)
    if not text:
        return ""
    if HAS_UNDERTHESEA:
        try:
            return word_tokenize(text, format="text")
        except Exception:
            return text
    return text


def prepare_phobert_text(text: str) -> str:
    text = clean_text_for_phobert(text)
    text = maybe_segment_vi_text(text)
    return text if text else "[EMPTY]"


def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)


def cosine_similarity_matrix(query_emb, doc_embs):
    return np.dot(query_emb, doc_embs.T)

In [ ]:
if RUN_EMBEDDING:
    tokenizer = AutoTokenizer.from_pretrained(PHOBERT_MODEL_NAME)
    model = AutoModel.from_pretrained(PHOBERT_MODEL_NAME)
    model.to(DEVICE)
    model.eval()

    def encode_phobert_texts(texts, batch_size=PHOBERT_BATCH_SIZE, max_length=PHOBERT_MAX_LENGTH_MATCH):
        embeddings = []
        prepared = [prepare_phobert_text(t) for t in texts]

        for i in tqdm(range(0, len(prepared), batch_size), desc="Encoding"):
            batch = prepared[i:i + batch_size]
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            input_ids = encoded["input_ids"].to(DEVICE)
            attention_mask = encoded["attention_mask"].to(DEVICE)

            with torch.no_grad():
                output = model(input_ids=input_ids, attention_mask=attention_mask)

            batch_emb = mean_pooling(output, attention_mask)

            if NORMALIZE_EMBEDDINGS:
                batch_emb = F.normalize(batch_emb, p=2, dim=1)

            embeddings.append(batch_emb.cpu().numpy())

        return np.vstack(embeddings)

    print("[INFO] PhoBERT loaded successfully.")
else:
    print("[INFO] RUN_EMBEDDING=False -> skip model loading.")

In [ ]:
job_dense_embeddings = None

df_clean["embedding_row_id"] = np.arange(len(df_clean))

if RUN_EMBEDDING:
    job_dense_embeddings = encode_phobert_texts(
        df_clean["job_text_phobert_match"].fillna("").tolist(),
        batch_size=PHOBERT_BATCH_SIZE,
        max_length=PHOBERT_MAX_LENGTH_MATCH,
    )
    df_clean["has_dense_embedding"] = 1
    print("job_dense_embeddings shape:", job_dense_embeddings.shape)
else:
    df_clean["has_dense_embedding"] = 0
    print("[INFO] Skip job-level embedding.")

In [ ]:
section_dense_embeddings = None

if RUN_EMBEDDING and RUN_SECTION_EMBEDDING and len(job_sections_df) > 0:
    job_sections_df["section_embedding_row_id"] = np.arange(len(job_sections_df))
    section_dense_embeddings = encode_phobert_texts(
        job_sections_df["chunk_text_phobert"].fillna("").tolist(),
        batch_size=PHOBERT_BATCH_SIZE,
        max_length=PHOBERT_MAX_LENGTH_CHUNK,
    )
    job_sections_df["has_dense_embedding"] = 1
    print("section_dense_embeddings shape:", section_dense_embeddings.shape)
else:
    job_sections_df["has_dense_embedding"] = 0
    print("[INFO] Skip section-level embedding.")

In [ ]:
def encode_query_for_matching(query: str, max_length=PHOBERT_MAX_LENGTH_MATCH):
    q = encode_phobert_texts([query], batch_size=1, max_length=max_length)
    return q[0]


def retrieve_top_jobs(query: str, top_k: int = 10):
    if job_dense_embeddings is None:
        raise RuntimeError("job_dense_embeddings is None. Hãy bật RUN_EMBEDDING=True")

    q_emb = encode_query_for_matching(query, max_length=PHOBERT_MAX_LENGTH_MATCH)
    scores = job_dense_embeddings @ q_emb
    top_idx = np.argsort(scores)[::-1][:top_k]

    cols = [
        "job_url", "job_title_display", "job_family", "location_norm",
        "work_mode", "skills_required", "skills_preferred", "job_text_phobert_match"
    ]
    out = df_clean.iloc[top_idx][cols].copy()
    out["cosine_score"] = scores[top_idx]
    return out.reset_index(drop=True)


def retrieve_top_sections(query: str, top_k: int = 10):
    if section_dense_embeddings is None:
        raise RuntimeError("section_dense_embeddings is None. Hãy bật RUN_SECTION_EMBEDDING=True")

    q_emb = encode_query_for_matching(query, max_length=PHOBERT_MAX_LENGTH_CHUNK)
    scores = section_dense_embeddings @ q_emb
    top_idx = np.argsort(scores)[::-1][:top_k]

    cols = [
        "job_url", "job_title_display", "job_family", "section_type",
        "chunk_order", "section_priority", "chunk_text_raw"
    ]
    out = job_sections_df.iloc[top_idx][cols].copy()
    out["cosine_score"] = scores[top_idx]
    return out.reset_index(drop=True)


# Demo
demo_query = "Data Analyst cần SQL Power BI và kinh nghiệm phân tích dữ liệu"
try:
    display(retrieve_top_jobs(demo_query, top_k=5))
    display(retrieve_top_sections(demo_query, top_k=5))
except Exception as e:
    print("[INFO] Demo retrieval skipped:", e)

In [ ]:
DOWNSTREAM_FIELD_GUIDE = {
    "tfidf_input": "job_text_sparse",
    "phobert_matching_input": "job_text_phobert_match",
    "phobert_chatbot_input": "job_text_phobert_chatbot",
    "chatbot_chunk_table": "job_sections_df",
    "chatbot_chunk_text_field": "chunk_text_phobert",
    "skill_table": "job_skill_map_df",
    "role_skill_stats": "role_skill_stats_df",
    "retrieval_metric": "cosine_similarity_on_l2_normalized_embeddings",
}

pd.Series(DOWNSTREAM_FIELD_GUIDE)

In [ ]:
matching_cols = [
    "job_url", "job_id",
    "job_title_display", "job_title_canonical", "job_family",
    "location_norm", "location_city", "location_district", "work_mode",
    "salary_min_vnd_month", "salary_max_vnd_month", "salary_is_negotiable",
    "experience_min_years", "experience_max_years", "experience_type",
    "education_level_norm", "employment_type_norm", "job_level_norm", "seniority_from_title",
    "skills_extracted", "skills_required", "skills_preferred", "skill_groups",
    "job_text_sparse", "job_text_phobert_match", "job_text_phobert_chatbot",
    "embedding_row_id", "has_dense_embedding", "dense_encoder_route", "dense_similarity_metric",
]

matching_cols = [c for c in matching_cols if c in df_clean.columns]
df_matching_ready = df_clean[matching_cols].copy()
df_matching_ready["dense_model_name"] = PHOBERT_MODEL_NAME
df_matching_ready["dense_similarity_metric"] = "cosine"

display(df_matching_ready.head(3))
print(df_matching_ready.shape)

In [ ]:
chatbot_cols = [
    "job_url", "job_id",
    "job_title_display", "job_title_canonical", "job_family",
    "location_norm", "work_mode",
    "salary_min_vnd_month", "salary_max_vnd_month",
    "experience_min_years", "experience_max_years",
    "skills_extracted", "skills_required", "skills_preferred",
    "requirements_clean_phobert", "description_clean_phobert", "benefits_clean_phobert",
    "job_text_phobert_chatbot",
    "embedding_row_id", "has_dense_embedding"
]

chatbot_cols = [c for c in chatbot_cols if c in df_clean.columns]
df_chatbot_ready = df_clean[chatbot_cols].copy()

display(df_chatbot_ready.head(3))
print(df_chatbot_ready.shape)

In [ ]:
section_cols = [
    "section_embedding_row_id",
    "job_url", "job_title_display", "job_title_canonical",
    "job_family", "location_norm", "work_mode",
    "section_type", "chunk_order", "section_priority",
    "chunk_text_raw", "chunk_text_phobert",
    "has_dense_embedding"
]
section_cols = [c for c in section_cols if c in job_sections_df.columns]

job_sections_ready = job_sections_df[section_cols].copy()
display(job_sections_ready.head(3))
print(job_sections_ready.shape)

In [ ]:
artifact_paths = {}

artifact_paths["jobs_matching_ready_v3"] = save_table(
    df_matching_ready, ARTIFACT_DIR / "jobs_matching_ready_v3"
)

artifact_paths["jobs_chatbot_ready_v3"] = save_table(
    df_chatbot_ready, ARTIFACT_DIR / "jobs_chatbot_ready_v3"
)

artifact_paths["jobs_chatbot_sections_v3"] = save_table(
    job_sections_ready, ARTIFACT_DIR / "jobs_chatbot_sections_v3"
)

artifact_paths["job_skill_map_v3"] = save_table(
    job_skill_map_df, ARTIFACT_DIR / "job_skill_map_v3"
)

artifact_paths["role_skill_stats_v3"] = save_table(
    role_skill_stats_df, ARTIFACT_DIR / "role_skill_stats_v3"
)

artifact_paths["job_embedding_index_v3"] = save_table(
    df_clean[["embedding_row_id", "job_url", "job_title_display", "job_text_phobert_match"]],
    ARTIFACT_DIR / "job_embedding_index_v3"
)

if len(job_sections_df) > 0:
    artifact_paths["job_section_embedding_index_v3"] = save_table(
        job_sections_df[
            ["section_embedding_row_id", "job_url", "job_title_display", "section_type", "chunk_order", "chunk_text_raw"]
        ],
        ARTIFACT_DIR / "job_section_embedding_index_v3"
    )

if RUN_EMBEDDING and job_dense_embeddings is not None:
    job_emb_path = ARTIFACT_DIR / "job_dense_embeddings_v3.npy"
    np.save(job_emb_path, job_dense_embeddings)
    artifact_paths["job_dense_embeddings_v3"] = str(job_emb_path)

if RUN_EMBEDDING and RUN_SECTION_EMBEDDING and section_dense_embeddings is not None:
    section_emb_path = ARTIFACT_DIR / "section_dense_embeddings_v3.npy"
    np.save(section_emb_path, section_dense_embeddings)
    artifact_paths["section_dense_embeddings_v3"] = str(section_emb_path)

print("[INFO] Saved main artifacts.")
pd.Series(artifact_paths)

In [ ]:
manifest = {
    "notebook_version": NOTEBOOK_VERSION,
    "run_timestamp": datetime.now().isoformat(),
    "raw_input_path": str(RAW_INPUT_PATH),
    "artifact_dir": str(ARTIFACT_DIR.resolve()),
    "n_jobs": int(len(df_clean)),
    "n_sections": int(len(job_sections_df)),
    "embedding_config": {
        "model_name": PHOBERT_MODEL_NAME,
        "metric": "cosine",
        "normalize_embeddings": NORMALIZE_EMBEDDINGS,
        "job_text_field": "job_text_phobert_match",
        "chatbot_text_field": "job_text_phobert_chatbot",
        "section_text_field": "chunk_text_phobert",
        "max_length_match": PHOBERT_MAX_LENGTH_MATCH,
        "max_length_chatbot": PHOBERT_MAX_LENGTH_CHATBOT,
        "max_length_chunk": PHOBERT_MAX_LENGTH_CHUNK,
        "batch_size": PHOBERT_BATCH_SIZE,
        "segmentation": "underthesea_if_available",
    },
    "downstream_field_guide": DOWNSTREAM_FIELD_GUIDE,
    "artifacts": artifact_paths,
}

manifest_path = ARTIFACT_DIR / "manifest_v3_phobert.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("[INFO] Manifest saved:", manifest_path)
display(pd.Series(manifest))

In [ ]:
length_stats = pd.DataFrame({
    "job_text_sparse_len": df_clean["job_text_sparse"].fillna("").str.len(),
    "job_text_phobert_match_len": df_clean["job_text_phobert_match"].fillna("").str.len(),
    "job_text_phobert_chatbot_len": df_clean["job_text_phobert_chatbot"].fillna("").str.len(),
})

display(length_stats.describe())

if len(job_sections_df) > 0:
    section_stats = (
        job_sections_df.assign(chunk_len=job_sections_df["chunk_text_phobert"].fillna("").str.len())
        .groupby("section_type")["chunk_len"]
        .describe()
        .reset_index()
    )
    display(section_stats)

print("Avg skills per job:", df_clean["skills_extracted"].apply(len).mean())
print("Jobs without extracted skills:", (df_clean["skills_extracted"].apply(len) == 0).sum())

## Ghi chú sử dụng artifact

- `jobs_matching_ready_v3`: dùng cho matching/recommendation
  - trường chính: `job_text_phobert_match`
  - vector tương ứng: `job_dense_embeddings_v3.npy`

- `jobs_chatbot_ready_v3`: dùng cho chatbot summary-level
  - trường chính: `job_text_phobert_chatbot`

- `jobs_chatbot_sections_v3`: dùng cho retrieval theo chunk/section
  - trường chính: `chunk_text_phobert`
  - vector tương ứng: `section_dense_embeddings_v3.npy`

- `job_skill_map_v3`: bảng long format job-skill
- `role_skill_stats_v3`: thống kê skill theo nhóm nghề
- `manifest_v3_phobert.json`: metadata toàn bộ pipeline